# Sentinel-2 Crop Health Monitoring: Seven Spectral Indices

## 1. Project Introduction

This notebook analyzes the existing six-band Sentinel-2 surface-reflectance GeoTIFF for Mysuru, Karnataka, India. It calculates seven vegetation and moisture indices, summarizes them, creates a seven-panel map, and exports one float32 GeoTIFF per index.

**Scene:** 6 February 2025 | **Collection:** `COPERNICUS/S2_SR_HARMONIZED` | **Grid:** 20 m | **Project CRS designation:** EPSG:32643 (the source raster CRS is copied unchanged).

| Index | Formula (reflectance bands) | Agricultural interpretation |
|---|---|---|
| NDVI | (B8 - B4) / (B8 + B4) | Canopy greenness and relative vegetation vigor. |
| EVI | 2.5 (B8 - B4) / (B8 + 6 B4 - 7.5 B2 + 1) | Canopy vigor with improved sensitivity in dense vegetation and reduced soil/atmospheric influence. |
| GNDVI | (B8 - B3) / (B8 + B3) | Greenness and chlorophyll-related variation; can be useful for crop vigor monitoring. |
| NDRE | (B8 - B5) / (B8 + B5) | Red-edge response for canopy chlorophyll and stress, including denser canopies. |
| NDMI | (B8 - B11) / (B8 + B11) | Canopy moisture variation and potential water stress. |
| MSI | B11 / B8 | Moisture stress indicator; larger values commonly indicate lower vegetation water content. |
| SAVI | 1.5 (B8 - B4) / (B8 + B4 + 0.5) | Greenness adjusted for soil background, useful where canopy cover is sparse. |

This workflow accepts a single-band-description-verified, six-band GeoTIFF in **B2, B3, B4, B5, B8, B11** order on a common 20 m grid. Edit the configuration cell for another compatible scene. It does not fetch imagery or infer missing bands.

## 2. Library Imports

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
import numpy as np
import pandas as pd
import rasterio
from rasterio.transform import array_bounds

print("Core libraries imported.")

Core libraries imported.


## 3. Project Directory Setup

Locate the existing project by walking upward from the notebook kernel working directory. This works when the notebook is opened from either the project root or its `notebooks` directory.

In [2]:
def find_project_root(start_dir):
    """Find the existing project root from the current notebook directory."""
    start_dir = Path(start_dir).resolve()
    for candidate in (start_dir, *start_dir.parents):
        if all((candidate / folder).is_dir() for folder in ("data", "notebooks", "outputs")):
            return candidate
    raise FileNotFoundError(
        f"Could not find a project root containing data, notebooks, and outputs above {start_dir}."
    )


PROJECT_ROOT = find_project_root(Path.cwd())
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
INDEX_OUTPUT_DIR = OUTPUT_DIR / "all_7_indices"
GEOTIFF_DIR = INDEX_OUTPUT_DIR / "geotiff"

# Change these scene settings to analyze a compatible six-band, 20 m Sentinel-2 GeoTIFF.
INPUT_RASTER = DATA_DIR / "sentinel2_complete_bands" / "Mysuru_Sentinel2_6_Bands.tif"
STUDY_AREA = "Mysuru, Karnataka, India"
ACQUISITION_DATE = "2025-02-06"
SENTINEL_COLLECTION = "COPERNICUS/S2_SR_HARMONIZED"
CRS_LABEL = "EPSG:32643 (source CRS retained unchanged)"
EXPECTED_BANDS = ("B2", "B3", "B4", "B5", "B8", "B11")
EXPECTED_RESOLUTION_M = 20.0
REFLECTANCE_SCALE = 10000.0
EVI_DENOMINATOR_THRESHOLD = 0.1
NODATA_VALUE = -9999.0

INDEX_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
GEOTIFF_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Input raster:", INPUT_RASTER)
print("Output directory:", INDEX_OUTPUT_DIR)

Project root: C:\Users\T RAO\OneDrive\Desktop\Crop_Health_Monitoring
Input raster: C:\Users\T RAO\OneDrive\Desktop\Crop_Health_Monitoring\data\sentinel2_complete_bands\Mysuru_Sentinel2_6_Bands.tif
Output directory: C:\Users\T RAO\OneDrive\Desktop\Crop_Health_Monitoring\outputs\all_7_indices


## 4. Satellite Data Information

The settings below describe the supplied scene. Surface-reflectance integer values are converted to unit reflectance using the Sentinel-2 scale factor of 10,000. Values are not clipped after conversion or index calculation.

In [3]:
print("Study area:", STUDY_AREA)
print("Acquisition date:", ACQUISITION_DATE)
print("Sentinel-2 collection:", SENTINEL_COLLECTION)
print("Expected band order:", ", ".join(EXPECTED_BANDS))
print("Expected pixel size:", f"{EXPECTED_RESOLUTION_M:g} m")
print("Configured CRS designation:", CRS_LABEL)
print("Reflectance conversion: stored DN /", f"{REFLECTANCE_SCALE:g}")

Study area: Mysuru, Karnataka, India
Acquisition date: 2025-02-06
Sentinel-2 collection: COPERNICUS/S2_SR_HARMONIZED
Expected band order: B2, B3, B4, B5, B8, B11
Expected pixel size: 20 m
Configured CRS designation: EPSG:32643 (source CRS retained unchanged)
Reflectance conversion: stored DN / 10000


## 5. Input Data Validation

Validate the configured file, band count and descriptions, source CRS, dimensions, and 20 m pixel size before reading pixel arrays. The CRS is taken directly from the source; no EPSG database lookup or CRS reconstruction is used.

In [4]:
if not INPUT_RASTER.is_file():
    raise FileNotFoundError(f"Configured input GeoTIFF does not exist: {INPUT_RASTER}")

with rasterio.open(INPUT_RASTER) as source:
    if source.count != len(EXPECTED_BANDS):
        raise ValueError(
            f"Expected {len(EXPECTED_BANDS)} bands {EXPECTED_BANDS}, but found {source.count}."
        )
    if tuple(source.descriptions) != EXPECTED_BANDS:
        raise ValueError(
            "Band descriptions/order must be exactly "
            f"{EXPECTED_BANDS}; found {source.descriptions}."
        )
    if source.crs is None:
        raise ValueError("The source raster has no CRS definition.")
    if source.width <= 0 or source.height <= 0:
        raise ValueError("The source raster has invalid dimensions.")
    if not all(np.isclose(size, EXPECTED_RESOLUTION_M) for size in source.res):
        raise ValueError(
            f"Expected {EXPECTED_RESOLUTION_M:g} m pixels; found {source.res}."
        )

    source_profile = source.profile.copy()
    source_crs = source.crs
    source_transform = source.transform
    raster_width = source.width
    raster_height = source.height
    raster_resolution = source.res
    source_bounds = tuple(source.bounds)
    source_descriptions = tuple(source.descriptions)

print("Input validation passed.")
print(f"Dimensions: {raster_width} columns x {raster_height} rows")
print("Band descriptions:", source_descriptions)
print("Source CRS copied as read:", source_crs)
print("Pixel size:", raster_resolution, "metres")
print("Bounds:", source_bounds)

Input validation passed.
Dimensions: 498 columns x 499 rows
Band descriptions: ('B2', 'B3', 'B4', 'B5', 'B8', 'B11')
Source CRS copied as read: PROJCS["WGS 84 / UTM zone 43N",GEOGCS["WGS 84",DATUM["World Geodetic System 1984",SPHEROID["WGS 84",6378137,298.257223563]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]]],PROJECTION["Transverse_Mercator"],PARAMETER["latitude_of_origin",0],PARAMETER["central_meridian",75],PARAMETER["scale_factor",0.9996],PARAMETER["false_easting",500000],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH]]
Pixel size: (20.0, 20.0) metres
Bounds: (673320.0, 1354820.0, 683280.0, 1364800.0)


## 6. Band Loading

Read all six bands together to preserve their common pixel grid. Any explicit source mask/NoData is retained as a mask for the later valid-pixel calculation.

In [5]:
with rasterio.open(INPUT_RASTER) as source:
    masked_source = source.read(masked=True).astype(np.float32)
    source_mask = np.ma.getmaskarray(masked_source)
    raw_bands = masked_source.filled(np.nan)

raw_band_arrays = {
    band_name: raw_bands[band_number]
    for band_number, band_name in enumerate(source_descriptions)
}

print("Loaded band array shape:", raw_bands.shape)
print("Stored data type:", raw_bands.dtype)
for band_name, band_array in raw_band_arrays.items():
    print(f"{band_name}: {band_array.shape}")

Loaded band array shape: (6, 499, 498)
Stored data type: float32
B2: (499, 498)
B3: (499, 498)
B4: (499, 498)
B5: (499, 498)
B8: (499, 498)
B11: (499, 498)


## 7. Reflectance Conversion

The source stores scaled surface reflectance. Convert each band to unit reflectance (DN / 10,000) before applying formulas that include constants such as the EVI coefficients and SAVI soil factor.

In [6]:
reflectance = {
    band_name: band_array / REFLECTANCE_SCALE
    for band_name, band_array in raw_band_arrays.items()
}

for band_name, band_array in reflectance.items():
    finite_values = band_array[np.isfinite(band_array)]
    print(
        f"{band_name}: min={finite_values.min():.4f}, "
        f"max={finite_values.max():.4f}, mean={finite_values.mean():.4f}"
    )

B2: min=0.0000, max=1.1395, mean=0.0385
B3: min=0.0000, max=1.1610, mean=0.0471
B4: min=0.0000, max=1.0542, mean=0.0537
B5: min=0.0000, max=0.9035, mean=0.0680
B8: min=0.0000, max=1.2802, mean=0.1027
B11: min=0.0000, max=0.9023, mean=0.1139


## 8. Data Quality Assessment

The supplied raster has no declared NoData value, so the existing rule that requires positive, finite reflectance in every input band is preserved. An explicit source mask is also honored. Zero-filled pixels are treated as invalid; reflectance above 1 is retained rather than clipped.

In [7]:
for band_name, band_array in reflectance.items():
    finite_values = band_array[np.isfinite(band_array)]
    print(
        f"{band_name}: zero pixels={np.count_nonzero(band_array == 0)}, "
        f"non-finite pixels={np.count_nonzero(~np.isfinite(band_array))}, "
        f"positive p2/p50/p98={np.percentile(finite_values[finite_values > 0], [2, 50, 98]).round(4)}"
    )

B2: zero pixels=140207, non-finite pixels=0, positive p2/p50/p98=[0.0347 0.0804 0.181 ]
B3: zero pixels=140207, non-finite pixels=0, positive p2/p50/p98=[0.05   0.1021 0.2028]
B4: zero pixels=140218, non-finite pixels=0, positive p2/p50/p98=[0.0385 0.1206 0.2395]
B5: zero pixels=140207, non-finite pixels=0, positive p2/p50/p98=[0.0817 0.1533 0.2648]
B8: zero pixels=140207, non-finite pixels=0, positive p2/p50/p98=[0.1438 0.2291 0.3851]
B11: zero pixels=140207, non-finite pixels=0, positive p2/p50/p98=[0.1564 0.257  0.4091]


## 9. Valid Pixel Masking

A pixel is valid only when all six bands are unmasked, finite, and greater than zero. This preserves the notebook's original masking policy. EVI applies its additional denominator-stability mask during index calculation.

In [8]:
reflectance_stack = np.stack([reflectance[name] for name in EXPECTED_BANDS])
valid_mask = (
    ~np.any(source_mask, axis=0)
    & np.all(np.isfinite(reflectance_stack), axis=0)
    & np.all(reflectance_stack > 0, axis=0)
)

total_pixel_count = valid_mask.size
base_valid_pixel_count = int(np.count_nonzero(valid_mask))
base_invalid_pixel_count = total_pixel_count - base_valid_pixel_count

print("Total pixels:", total_pixel_count)
print("Valid pixels before index-specific denominator checks:", base_valid_pixel_count)
print("Invalid pixels:", base_invalid_pixel_count)
print("Valid share:", f"{100 * base_valid_pixel_count / total_pixel_count:.2f}%")

Total pixels: 248502
Valid pixels before index-specific denominator checks: 108284
Invalid pixels: 140218
Valid share: 43.57%


## 10. Spectral Index Calculation

Each ratio uses a finite, non-zero denominator and writes invalid results as NaN. The EVI denominator retains the existing stability rule: pixels with `abs(denominator) < 0.1` are masked. No scientific index values are clipped.

In [9]:
blue = reflectance["B2"]
green = reflectance["B3"]
red = reflectance["B4"]
red_edge = reflectance["B5"]
nir = reflectance["B8"]
swir1 = reflectance["B11"]


def safe_ratio(numerator, denominator, pixel_mask):
    """Calculate a ratio only where inputs are finite and the denominator is non-zero."""
    result = np.full(denominator.shape, np.nan, dtype=np.float32)
    safe_pixels = (
        pixel_mask
        & np.isfinite(numerator)
        & np.isfinite(denominator)
        & (denominator != 0)
    )
    np.divide(numerator, denominator, out=result, where=safe_pixels)
    result[~np.isfinite(result)] = np.nan
    return result


ndvi = safe_ratio(nir - red, nir + red, valid_mask)
evi_denominator = nir + 6 * red - 7.5 * blue + 1
evi_mask = valid_mask & (np.abs(evi_denominator) >= EVI_DENOMINATOR_THRESHOLD)
evi = safe_ratio(2.5 * (nir - red), evi_denominator, evi_mask)
gndvi = safe_ratio(nir - green, nir + green, valid_mask)
ndre = safe_ratio(nir - red_edge, nir + red_edge, valid_mask)
ndmi = safe_ratio(nir - swir1, nir + swir1, valid_mask)
msi = safe_ratio(swir1, nir, valid_mask)
savi = safe_ratio(1.5 * (nir - red), nir + red + 0.5, valid_mask)

indices = {
    "NDVI": ndvi,
    "EVI": evi,
    "GNDVI": gndvi,
    "NDRE": ndre,
    "NDMI": ndmi,
    "MSI": msi,
    "SAVI": savi,
}

for index_name, index_array in indices.items():
    valid_count = int(np.count_nonzero(np.isfinite(index_array)))
    if valid_count == 0:
        raise ValueError(f"{index_name} has no valid pixels after masking.")
    print(f"{index_name}: calculated; {valid_count:,} finite pixels")
print(
    "EVI pixels excluded by denominator stability rule:",
    int(np.count_nonzero(valid_mask & (np.abs(evi_denominator) < EVI_DENOMINATOR_THRESHOLD)),),
)

NDVI: calculated; 108,284 finite pixels
EVI: calculated; 108,273 finite pixels
GNDVI: calculated; 108,284 finite pixels
NDRE: calculated; 108,284 finite pixels
NDMI: calculated; 108,284 finite pixels
MSI: calculated; 108,284 finite pixels
SAVI: calculated; 108,284 finite pixels
EVI pixels excluded by denominator stability rule: 11


## 11. Index Statistics

Compute statistics from finite index pixels only. The standard deviation uses the population convention (`ddof=0`), consistent with the earlier notebook. Invalid-pixel counts include both the base scene mask and any index-specific denominator exclusions.

In [10]:
statistics_rows = []
for index_name, index_array in indices.items():
    values = index_array[np.isfinite(index_array)]
    statistics_rows.append(
        {
            "Index Name": index_name,
            "Minimum": float(np.min(values)),
            "Maximum": float(np.max(values)),
            "Mean": float(np.mean(values)),
            "Median": float(np.median(values)),
            "Standard Deviation": float(np.std(values, ddof=0)),
            "Valid Pixel Count": int(values.size),
            "Invalid Pixel Count": int(index_array.size - values.size),
        }
    )

statistics = pd.DataFrame(statistics_rows)
statistics_path = INDEX_OUTPUT_DIR / "index_statistics.csv"
statistics.to_csv(statistics_path, index=False, float_format="%.8f")
print(statistics.round(4).to_string(index=False))
print("Statistics saved:", statistics_path)

Index Name  Minimum  Maximum    Mean  Median  Standard Deviation  Valid Pixel Count  Invalid Pixel Count
      NDVI  -0.1959   0.9843  0.3267  0.2978              0.1910             108284               140218
       EVI  -4.2086   4.0271  0.2140  0.1919              0.1356             108273               140229
     GNDVI  -0.5241   0.8180  0.3756  0.3907              0.1643             108284               140218
      NDRE  -0.3628   0.7390  0.2047  0.1876              0.1366             108284               140218
      NDMI  -0.3882   0.6136 -0.0498 -0.0712              0.1130             108284               140218
       MSI   0.2394   2.2689  1.1311  1.1534              0.2264             108284               140218
      SAVI  -0.0727   0.7686  0.1973  0.1842              0.1151             108284               140218
Statistics saved: C:\Users\T RAO\OneDrive\Desktop\Crop_Health_Monitoring\outputs\all_7_indices\index_statistics.csv


## 12. Seven-Index Visualization

The figure uses a 2 x 4 layout with one panel and color bar per index. Invalid pixels are gray. Per-index 2nd and 98th percentile color limits improve visual contrast only; the arrays and GeoTIFF exports remain un-clipped.

In [11]:
color_maps = {
    "NDVI": "RdYlGn",
    "EVI": "RdYlGn",
    "GNDVI": "RdYlGn",
    "NDRE": "RdYlGn",
    "NDMI": "BrBG",
    "MSI": "YlOrRd",
    "SAVI": "RdYlGn",
}
index_titles = {
    "NDVI": "Greenness",
    "EVI": "Canopy vigor",
    "GNDVI": "Green vigor",
    "NDRE": "Red-edge response",
    "NDMI": "Moisture",
    "MSI": "Moisture stress",
    "SAVI": "Soil-adjusted greenness",
}
left, bottom, right, top = array_bounds(raster_height, raster_width, source_transform)
map_extent = (left, right, bottom, top)

fig, axes = plt.subplots(2, 4, figsize=(22, 12))
for axis, (index_name, index_array) in zip(axes.flat, indices.items()):
    valid_values = index_array[np.isfinite(index_array)]
    display_min, display_max = np.percentile(valid_values, [2, 98])
    if np.isclose(display_min, display_max):
        padding = max(abs(float(display_min)) * 0.01, 1e-6)
        display_min -= padding
        display_max += padding

    color_map = plt.get_cmap(color_maps[index_name]).copy()
    color_map.set_bad(color="#d9d9d9")
    image = axis.imshow(
        np.ma.masked_invalid(index_array),
        cmap=color_map,
        vmin=display_min,
        vmax=display_max,
        extent=map_extent,
        interpolation="nearest",
    )
    axis.set_title(f"{index_name} | {index_titles[index_name]}", fontsize=10, weight="bold")
    axis.set_xlabel("Easting (m)")
    axis.set_ylabel("Northing (m)")
    axis.set_aspect("equal")
    axis.xaxis.set_major_locator(MaxNLocator(nbins=4))
    axis.yaxis.set_major_locator(MaxNLocator(nbins=4))
    axis.tick_params(labelsize=8)
    colorbar = fig.colorbar(image, ax=axis, shrink=0.82, pad=0.03)
    colorbar.set_label(f"{index_name} value", fontsize=8)
    colorbar.ax.tick_params(labelsize=8)

axes.flat[7].axis("off")
axes.flat[7].text(
    0.05,
    0.82,
    "Map notes",
    fontsize=13,
    weight="bold",
    transform=axes.flat[7].transAxes,
)
axes.flat[7].text(
    0.05,
    0.68,
    "Gray pixels: invalid or masked\n"
    "Colors stretched to each index's 2nd-98th percentiles.\n"
    "Raster values are not clipped.",
    fontsize=11,
    va="top",
    transform=axes.flat[7].transAxes,
)
fig.suptitle(
    "Seven Sentinel-2 Crop Health Indices\n"
    f"{STUDY_AREA} | {ACQUISITION_DATE} | {EXPECTED_RESOLUTION_M:g} m",
    fontsize=16,
    weight="bold",
)
fig.tight_layout(rect=(0, 0.02, 1, 0.92))
visualization_path = INDEX_OUTPUT_DIR / "sentinel2_all_7_indices.png"
fig.savefig(visualization_path, dpi=240, bbox_inches="tight", facecolor="white")
plt.close(fig)
print("Seven-index visualization saved:", visualization_path)

Seven-index visualization saved: C:\Users\T RAO\OneDrive\Desktop\Crop_Health_Monitoring\outputs\all_7_indices\sentinel2_all_7_indices.png


## 13. GeoTIFF Export

Export one single-band Float32 GeoTIFF per index. Metadata is derived from the source profile so the source CRS, transform, width, and height remain unchanged. NaN results are written with the explicit NoData value `-9999`.

In [12]:
export_profile = source_profile.copy()
export_profile.update(
    driver="GTiff",
    count=1,
    dtype="float32",
    nodata=NODATA_VALUE,
    compress="lzw",
)

for index_name, index_array in indices.items():
    output_path = GEOTIFF_DIR / f"{index_name}.tif"
    output_array = np.where(
        np.isfinite(index_array), index_array, NODATA_VALUE
    ).astype(np.float32)
    with rasterio.open(output_path, "w", **export_profile) as destination:
        destination.write(output_array, 1)
        destination.set_band_description(1, index_name)
        destination.update_tags(
            STUDY_AREA=STUDY_AREA,
            ACQUISITION_DATE=ACQUISITION_DATE,
            COLLECTION=SENTINEL_COLLECTION,
            INDEX_NAME=index_name,
        )
    print("Exported:", output_path)

index_descriptions = {
    "NDVI": "Canopy greenness and relative vegetation vigor",
    "EVI": "Enhanced canopy vigor; denominator stability threshold applied",
    "GNDVI": "Green-band vigor and chlorophyll-related variation",
    "NDRE": "Red-edge response for canopy chlorophyll and stress",
    "NDMI": "Canopy moisture variation and potential water stress",
    "MSI": "Moisture stress ratio; larger values commonly indicate drier vegetation",
    "SAVI": "Soil-adjusted greenness for sparse vegetation",
}
project_summary_path = INDEX_OUTPUT_DIR / "project_summary.txt"
summary_lines = [
    "Sentinel-2 Crop Health Monitoring - Project Summary",
    "===================================================",
    f"Study area: {STUDY_AREA}",
    f"Acquisition date: {ACQUISITION_DATE}",
    f"Satellite: Sentinel-2",
    f"Google Earth Engine collection: {SENTINEL_COLLECTION}",
    f"CRS designation: {CRS_LABEL}",
    "CRS handling: source raster CRS copied unchanged; no EPSG reconstruction performed.",
    f"Spatial resolution: {EXPECTED_RESOLUTION_M:g} m",
    f"Raster dimensions: {raster_width} columns x {raster_height} rows",
    f"Input raster: {INPUT_RASTER}",
    f"Input bands (source order): {', '.join(EXPECTED_BANDS)}",
    f"Reflectance conversion: stored DN / {REFLECTANCE_SCALE:g}; values not clipped",
    "Valid-pixel rule: finite, positive reflectance in all six bands, honoring source masks.",
    f"EVI denominator stability rule: abs(denominator) >= {EVI_DENOMINATOR_THRESHOLD:g}.",
    "",
    "Band information:",
    "  B2: Blue",
    "  B3: Green",
    "  B4: Red",
    "  B5: Red Edge 1",
    "  B8: Near Infrared",
    "  B11: Shortwave Infrared 1",
    "",
    "Index descriptions and agricultural applications:",
]
summary_lines.extend(
    f"  {name}: {description}." for name, description in index_descriptions.items()
)
summary_lines.extend(
    [
        "",
        "Outputs:",
        f"  Visualization: {visualization_path}",
        f"  Statistics CSV: {statistics_path}",
        f"  Individual Float32 GeoTIFFs: {GEOTIFF_DIR}",
        f"  Project summary: {project_summary_path}",
        f"  GeoTIFF NoData value: {NODATA_VALUE:g}",
    ]
)
project_summary_path.write_text("\n".join(summary_lines) + "\n", encoding="utf-8")
print("Project summary saved:", project_summary_path)

Exported: C:\Users\T RAO\OneDrive\Desktop\Crop_Health_Monitoring\outputs\all_7_indices\geotiff\NDVI.tif


Exported: C:\Users\T RAO\OneDrive\Desktop\Crop_Health_Monitoring\outputs\all_7_indices\geotiff\EVI.tif


Exported: C:\Users\T RAO\OneDrive\Desktop\Crop_Health_Monitoring\outputs\all_7_indices\geotiff\GNDVI.tif


Exported: C:\Users\T RAO\OneDrive\Desktop\Crop_Health_Monitoring\outputs\all_7_indices\geotiff\NDRE.tif


Exported: C:\Users\T RAO\OneDrive\Desktop\Crop_Health_Monitoring\outputs\all_7_indices\geotiff\NDMI.tif


Exported: C:\Users\T RAO\OneDrive\Desktop\Crop_Health_Monitoring\outputs\all_7_indices\geotiff\MSI.tif


Exported: C:\Users\T RAO\OneDrive\Desktop\Crop_Health_Monitoring\outputs\all_7_indices\geotiff\SAVI.tif
Project summary saved: C:\Users\T RAO\OneDrive\Desktop\Crop_Health_Monitoring\outputs\all_7_indices\project_summary.txt


## 14. Output Verification

Reopen every generated file and verify the source grid and CRS are preserved, all seven bands/maps exist, NoData is declared, and the CSV and project report contain the expected deliverables.

In [13]:
expected_index_names = ("NDVI", "EVI", "GNDVI", "NDRE", "NDMI", "MSI", "SAVI")
assert visualization_path.is_file() and visualization_path.stat().st_size > 0
assert statistics_path.is_file() and statistics_path.stat().st_size > 0
assert project_summary_path.is_file() and project_summary_path.stat().st_size > 0

saved_statistics = pd.read_csv(statistics_path)
assert tuple(saved_statistics["Index Name"]) == expected_index_names
required_statistics_columns = {
    "Index Name", "Minimum", "Maximum", "Mean", "Median",
    "Standard Deviation", "Valid Pixel Count", "Invalid Pixel Count",
}
assert required_statistics_columns.issubset(saved_statistics.columns)

with rasterio.open(INPUT_RASTER) as source:
    assert source.count == 6
    assert tuple(source.descriptions) == EXPECTED_BANDS
    assert source.crs == source_crs
    assert source.transform == source_transform
    assert (source.width, source.height) == (raster_width, raster_height)
    assert source.res == raster_resolution

for index_name in expected_index_names:
    output_path = GEOTIFF_DIR / f"{index_name}.tif"
    assert output_path.is_file() and output_path.stat().st_size > 0
    with rasterio.open(output_path) as output:
        assert output.count == 1
        assert output.dtypes == ("float32",)
        assert output.nodata == NODATA_VALUE
        assert output.crs == source_crs
        assert output.transform == source_transform
        assert (output.width, output.height) == (raster_width, raster_height)
        assert output.res == raster_resolution
        assert output.descriptions == (index_name,)
        saved_values = output.read(1, masked=True)
        expected_mask = ~np.isfinite(indices[index_name])
        assert np.array_equal(np.ma.getmaskarray(saved_values), expected_mask)
        np.testing.assert_allclose(
            saved_values.filled(np.nan), indices[index_name], rtol=1e-6, atol=1e-7,
            equal_nan=True,
        )

summary_text = project_summary_path.read_text(encoding="utf-8")
assert STUDY_AREA in summary_text
assert ACQUISITION_DATE in summary_text
assert SENTINEL_COLLECTION in summary_text
assert all(index_name in summary_text for index_name in expected_index_names)

print("Input TIFF: PASS (6 expected bands; source CRS and grid recorded)")
print("Index calculations: PASS (7 indices; output pixel values checked)")
print("Visualization, statistics CSV, and project summary: PASS")
print("GeoTIFF CRS, transform, dimensions, resolution, dtype, NoData: PASS for all 7")
print("CRS equality is verified against the source raster; no EPSG lookup was required.")

Input TIFF: PASS (6 expected bands; source CRS and grid recorded)
Index calculations: PASS (7 indices; output pixel values checked)
Visualization, statistics CSV, and project summary: PASS
GeoTIFF CRS, transform, dimensions, resolution, dtype, NoData: PASS for all 7
CRS equality is verified against the source raster; no EPSG lookup was required.


## 15. Final Project Summary

The processing products are written under `outputs/all_7_indices/`. Input location, scene metadata, reflectance scaling, expected bands, and EVI stability threshold are configurable in Section 3. A replacement raster must satisfy the explicit six-band order and 20 m grid validation above.

In [14]:
print("PROJECT RUN COMPLETE")
print("Study area:", STUDY_AREA)
print("Acquisition date:", ACQUISITION_DATE)
print("Input:", INPUT_RASTER)
print("Visualization:", visualization_path)
print("Statistics:", statistics_path)
print("GeoTIFF directory:", GEOTIFF_DIR)
print("Project report:", project_summary_path)
print(f"Pixels: {raster_width} x {raster_height}; source resolution: {raster_resolution} m")
print("All mandatory output checks passed.")

PROJECT RUN COMPLETE
Study area: Mysuru, Karnataka, India
Acquisition date: 2025-02-06
Input: C:\Users\T RAO\OneDrive\Desktop\Crop_Health_Monitoring\data\sentinel2_complete_bands\Mysuru_Sentinel2_6_Bands.tif
Visualization: C:\Users\T RAO\OneDrive\Desktop\Crop_Health_Monitoring\outputs\all_7_indices\sentinel2_all_7_indices.png
Statistics: C:\Users\T RAO\OneDrive\Desktop\Crop_Health_Monitoring\outputs\all_7_indices\index_statistics.csv
GeoTIFF directory: C:\Users\T RAO\OneDrive\Desktop\Crop_Health_Monitoring\outputs\all_7_indices\geotiff
Project report: C:\Users\T RAO\OneDrive\Desktop\Crop_Health_Monitoring\outputs\all_7_indices\project_summary.txt
Pixels: 498 x 499; source resolution: (20.0, 20.0) m
All mandatory output checks passed.
